In [1]:
#WE import the libraries
import pandas as pd
import numpy as np

In [2]:
#loading our KCB modelling dataset
kcb=pd.read_csv(r"C:\Users\Win\NSE Project\Data\2013 to 2024 cosolidated_KCB_MODELLING_data.csv")
kcb.head()

,DATE,Day Price,Change%,12m Low,12m High,Day Low,Day High,Previous,Volume_log,MA_5,...,MA_12,STD_12,MA_50,STD_50,MA_200,STD_200,MA_5_vs_MA_50,MA_12_vs_MA_200,HighVol_5_vs_50,Daily_Range
0,2013-01-03,31.00,0.000098,33.75,55.5,30.25,31.50,30.25,13.580296,30.6250,...,30.625000,0.530330,30.625000,0.530330,30.625000,0.530330,0,0,0,1.25
1,2013-01-04,31.00,0.000194,33.75,55.5,31.00,31.50,31.00,15.476535,30.7500,...,30.750000,0.433013,30.750000,0.433013,30.750000,0.433013,0,0,0,0.50
2,2013-01-07,31.00,0.000190,33.75,55.5,30.25,31.50,31.00,14.527333,30.8125,...,30.812500,0.375000,30.812500,0.375000,30.812500,0.375000,0,0,0,1.25
3,2013-01-08,31.00,0.000467,33.75,55.5,30.25,31.25,31.00,13.365468,30.8500,...,30.850000,0.335410,30.850000,0.335410,30.850000,0.335410,0,0,0,1.00
4,2013-01-09,31.25,0.000357,33.75,55.5,31.00,32.00,31.00,14.321726,31.0500,...,30.916667,0.341565,30.916667,0.341565,30.916667,0.341565,1,0,0,1.00


In [3]:
# we comfirm that all our data is numerical
kcb.info()

<class 'pandas.DataFrame'>
RangeIndex: 2979 entries, 0 to 2978
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   DATE             2979 non-null   str    
 1   Day Price        2979 non-null   float64
 2   Change%          2979 non-null   float64
 3   12m Low          2979 non-null   float64
 4   12m High         2979 non-null   float64
 5   Day Low          2979 non-null   float64
 6   Day High         2979 non-null   float64
 7   Previous         2979 non-null   float64
 8   Volume_log       2979 non-null   float64
 9   MA_5             2979 non-null   float64
 10  STD_5            2979 non-null   float64
 11  MA_12            2979 non-null   float64
 12  STD_12           2979 non-null   float64
 13  MA_50            2979 non-null   float64
 14  STD_50           2979 non-null   float64
 15  MA_200           2979 non-null   float64
 16  STD_200          2979 non-null   float64
 17  MA_5_vs_MA_50    2979 non

In [4]:
# we engineer a new feature return
kcb['return'] = np.log(kcb['Day Price'] / kcb['Day Price'].shift(1))

In [5]:
# we engineer the target 
kcb['target'] = kcb['return'].shift(-1)

In [6]:
# Create lagged return features
kcb['lag1'] = kcb['return'].shift(1)
kcb['lag2'] = kcb['return'].shift(2)
kcb['lag3'] = kcb['return'].shift(3)
kcb['lag5'] = kcb['return'].shift(5)

In [7]:
#we drp the null values that emanated from engineering
kcb=kcb.dropna().reset_index(drop=True)
kcb.shape

(2972, 27)

In [8]:
kcb.head(2)

,DATE,Day Price,Change%,12m Low,12m High,Day Low,Day High,Previous,Volume_log,MA_5,...,MA_5_vs_MA_50,MA_12_vs_MA_200,HighVol_5_vs_50,Daily_Range,return,target,lag1,lag2,lag3,lag5
0,2013-01-11,32.75,0.0,33.75,55.5,31.5,33.25,31.75,15.086647,31.55,...,1,0,1,1.75,0.031010,0.015152,0.015873,0.008032,0.000000,0.0
1,2013-01-14,33.25,0.0,33.75,55.5,33.0,35.00,32.75,13.012104,32.00,...,1,0,1,2.00,0.015152,0.029632,0.031010,0.015873,0.008032,0.0


In [9]:
# we now define the features and the target
# we leave the return feature out so that the model doesnt see the future- preventing data leakage
feature_cols = ['Change%', 'Volume_log',
                'MA_5', 'MA_12', 'MA_50', 'MA_200',
                'STD_5', 'STD_12', 'STD_50', 'STD_200',
                'MA_5_vs_MA_50', 'MA_12_vs_MA_200', 'HighVol_5_vs_50',
                'Daily_Range',
                'lag1', 'lag2', 'lag3', 'lag5']

X = kcb[feature_cols]
y = kcb['target']

In [10]:
#we preview our features
X.head()

,Change%,Volume_log,MA_5,MA_12,MA_50,MA_200,STD_5,STD_12,STD_50,STD_200,MA_5_vs_MA_50,MA_12_vs_MA_200,HighVol_5_vs_50,Daily_Range,lag1,lag2,lag3,lag5
0,0.000000,15.086647,31.55,31.250000,31.250000,31.250000,0.737394,0.731925,0.731925,0.731925,1,0,1,1.75,0.015873,0.008032,0.000000,0.000000
1,0.000000,13.012104,32.00,31.472222,31.472222,31.472222,0.968246,0.955612,0.955612,0.955612,1,0,1,2.00,0.031010,0.015873,0.008032,0.000000
2,0.000089,13.925800,32.65,31.750000,31.750000,31.750000,1.193734,1.258306,1.258306,1.258306,1,0,0,1.50,0.015152,0.031010,0.015873,0.000000
3,0.000177,14.699898,33.30,32.000000,32.000000,32.000000,1.123610,1.453444,1.453444,1.453444,1,0,0,1.75,0.029632,0.015152,0.031010,0.008032
4,0.000000,13.431613,33.50,32.062500,32.062500,32.062500,0.829156,1.402615,1.402615,1.402615,1,0,0,2.25,0.007273,0.029632,0.015152,0.015873


In [11]:
# we preview our target
y.head()

0    0.015152
1    0.029632
2    0.007273
3   -0.052056
4    0.000000
Name: target, dtype: float64

## TRAIN_TEST_SPLIT

In [12]:
# WE split data in terms of time
#2013-2021 =train
#2022-2023= validate
#2024=test
train = kcb[kcb['DATE'] < '2022-01-01']
val   = kcb[(kcb['DATE'] >= '2022-01-01') & (kcb['DATE'] < '2024-01-01')]
test  = kcb[kcb['DATE'] >= '2024-01-01']

# Features and target split
X_train = train[feature_cols]
y_train = train['target']

X_val   = val[feature_cols]
y_val   = val['target']

X_test  = test[feature_cols]
y_test  = test['target']

In [13]:
X_train.head()

,Change%,Volume_log,MA_5,MA_12,MA_50,MA_200,STD_5,STD_12,STD_50,STD_200,MA_5_vs_MA_50,MA_12_vs_MA_200,HighVol_5_vs_50,Daily_Range,lag1,lag2,lag3,lag5
0,0.000000,15.086647,31.55,31.250000,31.250000,31.250000,0.737394,0.731925,0.731925,0.731925,1,0,1,1.75,0.015873,0.008032,0.000000,0.000000
1,0.000000,13.012104,32.00,31.472222,31.472222,31.472222,0.968246,0.955612,0.955612,0.955612,1,0,1,2.00,0.031010,0.015873,0.008032,0.000000
2,0.000089,13.925800,32.65,31.750000,31.750000,31.750000,1.193734,1.258306,1.258306,1.258306,1,0,0,1.50,0.015152,0.031010,0.015873,0.000000
3,0.000177,14.699898,33.30,32.000000,32.000000,32.000000,1.123610,1.453444,1.453444,1.453444,1,0,0,1.75,0.029632,0.015152,0.031010,0.008032
4,0.000000,13.431613,33.50,32.062500,32.062500,32.062500,0.829156,1.402615,1.402615,1.402615,1,0,0,2.25,0.007273,0.029632,0.015152,0.015873


In [14]:
X_train.tail()

,Change%,Volume_log,MA_5,MA_12,MA_50,MA_200,STD_5,STD_12,STD_50,STD_200,MA_5_vs_MA_50,MA_12_vs_MA_200,HighVol_5_vs_50,Daily_Range,lag1,lag2,lag3,lag5
2230,0.000013,10.451638,44.80,44.016667,43.931,43.88125,0.266927,0.800095,1.081943,2.476256,1,1,0,0.50,0.003346,0.002237,0.003365,0.002270
2231,0.000163,12.733168,44.89,44.200000,43.929,43.91025,0.210357,0.732679,1.079923,2.454436,1,1,0,0.80,0.006659,0.003346,0.002237,0.009029
2232,0.000152,13.303853,44.96,44.354167,43.927,43.93900,0.163554,0.684722,1.077801,2.433157,1,1,0,0.45,-0.005546,0.006659,0.003346,0.003365
2233,0.000284,11.202330,45.04,44.512500,43.934,43.96500,0.129422,0.623088,1.084701,2.418054,1,1,0,0.30,0.001112,-0.005546,0.006659,0.002237
2234,0.000080,13.169821,45.15,44.700000,43.951,43.99400,0.196850,0.522668,1.101876,2.400762,1,1,0,0.45,0.003328,0.001112,-0.005546,0.003346


## MODELLING

### 1. NAIVE PREDICTOR
- this acts as the base, that is, the upcoming models should outperform these metrics

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Naive prediction: tomorrow = yesterday
y_pred_naive = X_val['lag1']  # lag1 = yesterday's return

mae = mean_absolute_error(y_val, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_naive))  # compute RMSE manually
r2 = r2_score(y_val, y_pred_naive)

print("Naive Predictor Evaluation:")
print(f"MAE: {mae:.6f}, RMSE: {rmse:.6f}, R²: {r2:.4f}")

Naive Predictor Evaluation:
MAE: 0.014601, RMSE: 0.023641, R²: -0.8527
